# Pattern — Routing

**Routing** sends each incoming request to **one specialized handler** based on intent, rules, or a classifier. Unlike prompt chaining (fixed sequence for everyone) or ReAct (model picks tools in a loop), routing **branches once** at the start — then the chosen path runs its own pipeline.

```
  Customer message
       │
       ▼
  ┌──────────┐
  │  ROUTER  │  rules / classifier / embeddings
  └────┬─────┘
       │
   ┌───┼───┬─────────┬──────────┐
   ▼   ▼   ▼         ▼          ▼
 Order Policy Refund  Escalation  Fallback
 handler handler handler handler   FAQ
```

This notebook implements routing for **ShopCo Support Hub** and **ReleaseFlow** in plain Python: rule-based classification, confidence scores, fallback routes, hierarchical routing, and specialized handlers per branch.

**When to use:** distinct request types needing different prompts, tools, or SLAs.

**When not to use:** a single homogeneous task, or when the model must dynamically loop over tools (use ReAct instead).

In [ ]:
"""
Routing Pattern Lab — ShopCo + ReleaseFlow.
"""

import json
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable

USE_LIVE_LLM = False
CONFIDENCE_THRESHOLD = 0.55


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


def utcnow() -> str:
    return datetime.now(timezone.utc).isoformat()


ORDERS: dict[str, dict[str, Any]] = {
    "ORD-1001": {"order_id": "ORD-1001", "customer": "alice@shop.com", "status": "shipped", "total_usd": 1299.0},
    "ORD-1002": {"order_id": "ORD-1002", "customer": "bob@shop.com", "status": "processing", "total_usd": 449.0},
}

POLICIES: dict[str, str] = {
    "returns": "30-day returns on unused items. [pol-returns-01]",
    "shipping": "Free shipping over $50. [pol-shipping-02]",
    "refunds": "Escalate refunds over $1000. [pol-refunds-03]",
}

RELEASE_CHECKS: dict[str, dict[str, str]] = {
    "unit_tests": {"status": "pass", "detail": "412 tests passed"},
    "security_scan": {"status": "warn", "detail": "1 medium CVE"},
}

ROUTE_TRACE: list[dict[str, Any]] = []

print("Routing pattern lab ready")

---

## 1. Routing vs Chaining vs ReAct

| Pattern | Decision point | Number of paths |
|---------|----------------|-----------------|
| **Prompt chain** | None — same path for all | 1 fixed sequence |
| **Routing** | Once at entry | N branches, pick 1 |
| **ReAct** | Every step | Dynamic tool loop |

Routing is the thinnest way to add **specialization** without full multi-agent orchestration. Each route can internally use a prompt chain or single shot.

---

## 2. Route Model — Intent, Handler, Confidence

In [ ]:
@dataclass
class RouteDecision:
    route_id: str
    confidence: float
    reason: str
    scores: dict[str, float] = field(default_factory=dict)


@dataclass
class HandlerResult:
    route_id: str
    reply: str
    metadata: dict[str, Any] = field(default_factory=dict)


HandlerFn = Callable[[str, RouteDecision], HandlerResult]


@dataclass
class Route:
    route_id: str
    description: str
    handler: HandlerFn
    keywords: list[str] = field(default_factory=list)


print("Route model ready")

---

## 3. ShopCo Specialized Handlers

Each route owns a **focused handler** with its own tools and prompt logic — not one mega-prompt for everything.

In [ ]:
def handle_order_status(message: str, decision: RouteDecision) -> HandlerResult:
    oid = re.search(r"ORD-\d+", message.upper())
    if not oid:
        return HandlerResult("order_status", "Please provide your order ID (e.g. ORD-1001).")
    order = ORDERS.get(oid.group())
    if not order:
        return HandlerResult("order_status", f"Order {oid.group()} not found.")
    return HandlerResult(
        "order_status",
        f"{oid.group()} is {order['status']} (${order['total_usd']}).",
        {"order": order},
    )


def handle_policy(message: str, decision: RouteDecision) -> HandlerResult:
    msg = message.lower()
    if "return" in msg:
        return HandlerResult("policy", POLICIES["returns"])
    if "ship" in msg:
        return HandlerResult("policy", POLICIES["shipping"])
    return HandlerResult("policy", "Ask about returns, shipping, or refunds.")


def handle_refund(message: str, decision: RouteDecision) -> HandlerResult:
    oid = re.search(r"ORD-\d+", message.upper())
    if not oid:
        return HandlerResult("refund", "Refund requests need an order ID.")
    order = ORDERS.get(oid.group())
    if not order:
        return HandlerResult("refund", f"Order {oid.group()} not found.")
    reply = f"{POLICIES['refunds']} Order {oid.group()} ({order['status']})."
    if order["total_usd"] > 1000:
        reply += " Escalating to supervisor — amount exceeds $1000."
    return HandlerResult("refund", reply, {"escalated": order["total_usd"] > 1000})


def handle_escalation(message: str, decision: RouteDecision) -> HandlerResult:
    ticket = f"ESC-{uuid.uuid4().hex[:6].upper()}"
    return HandlerResult("escalation", f"Created escalation ticket {ticket}. A human agent will respond within 4 hours.", {"ticket": ticket})


def handle_fallback(message: str, decision: RouteDecision) -> HandlerResult:
    return HandlerResult(
        "fallback",
        "I can help with order status, returns policy, or refunds. What do you need?",
        {"low_confidence": decision.confidence},
    )


print("ShopCo handlers registered")

---

## 4. Rule-Based Router

Score each route by keyword overlap. Pick the highest score; tie-break by route priority.

In [ ]:
SHOPCO_ROUTES: list[Route] = [
    Route("escalation", "Human handoff", handle_escalation, ["manager", "supervisor", "human", "speak to someone"]),
    Route("refund", "Refund requests", handle_refund, ["refund", "money back", "chargeback"]),
    Route("order_status", "Order tracking", handle_order_status, ["status", "track", "ship", "delivery", "ord-"]),
    Route("policy", "Policy FAQ", handle_policy, ["policy", "return", "shipping", "warranty"]),
]


class RuleBasedRouter:
    def __init__(self, routes: list[Route], fallback_route_id: str = "fallback", threshold: float = CONFIDENCE_THRESHOLD):
        self.routes = routes
        self.fallback_route_id = fallback_route_id
        self.threshold = threshold
        self._handlers = {r.route_id: r.handler for r in routes}
        self._handlers["fallback"] = handle_fallback

    def classify(self, message: str) -> RouteDecision:
        msg = message.lower()
        scores: dict[str, float] = {}
        for route in self.routes:
            hits = sum(1 for kw in route.keywords if kw in msg)
            if hits:
                scores[route.route_id] = hits / len(route.keywords) + 0.15 * hits
        if re.search(r"ORD-\d+", message.upper()):
            scores["order_status"] = scores.get("order_status", 0) + 0.35
            if "refund" in msg:
                scores["refund"] = scores.get("refund", 0) + 0.45
        if any(kw in msg for kw in ("manager", "supervisor", "speak to someone")):
            scores["escalation"] = max(scores.get("escalation", 0), 0.85)
        if "policy" in msg or ("return" in msg and "refund" not in msg):
            scores["policy"] = max(scores.get("policy", 0), 0.65)
        if not scores:
            return RouteDecision("fallback", 0.0, "no keyword match", {})
        best_id = max(scores, key=scores.get)
        conf = min(1.0, scores[best_id])
        return RouteDecision(best_id, conf, "keyword scores", scores)

    def route_and_handle(self, message: str) -> tuple[RouteDecision, HandlerResult]:
        decision = self.classify(message)
        route_id = decision.route_id if decision.confidence >= self.threshold else self.fallback_route_id
        if route_id != decision.route_id:
            decision = RouteDecision(route_id, decision.confidence, f"below threshold → fallback", decision.scores)
        handler = self._handlers[route_id]
        result = handler(message, decision)
        ROUTE_TRACE.append({"message": message[:50], "route": route_id, "confidence": decision.confidence, "ts": utcnow()})
        return decision, result


router = RuleBasedRouter(SHOPCO_ROUTES)
d, r = router.route_and_handle("What's the status of ORD-1002?")
print(f"Route: {d.route_id} (conf={d.confidence:.2f}) → {r.reply}")

---

## 5. Route Matrix — Sample Messages

In [ ]:
test_messages = [
    "Where is ORD-1001? Has it shipped?",
    "What is your return policy?",
    "I want a refund for ORD-1001 — terrible service!",
    "Let me speak to a manager now.",
    "Hi there.",
]

print(f"{'Message':<45} {'Route':<14} Conf")
print("-" * 65)
for msg in test_messages:
    decision, result = router.route_and_handle(msg)
    print(f"{msg[:44]:<45} {decision.route_id:<14} {decision.confidence:.2f}")

---

## 6. Confidence Threshold and Fallback

Low-confidence classifications should not silently hit the wrong handler — route to **fallback** or ask a clarifying question.

In [ ]:
strict_router = RuleBasedRouter(SHOPCO_ROUTES, threshold=0.7)
ambiguous = "I have a question about my purchase."
d, r = strict_router.route_and_handle(ambiguous)
print(f"Ambiguous message → route={d.route_id}, conf={d.confidence:.2f}")
print(f"Reply: {r.reply}")

---

## 7. Hierarchical Routing — Two-Level

Large systems route **coarse intent first**, then **sub-intent**. ShopCo: `support` → `{order, billing, policy}`.

In [ ]:
class HierarchicalRouter:
    def __init__(self):
        self.l1_rules = {
            "support": ["order", "refund", "return", "ship", "ord-"],
            "escalation": ["manager", "human", "supervisor"],
            "general": [],
        }
        self.l2_routers = {
            "support": RuleBasedRouter(SHOPCO_ROUTES, threshold=0.4),
            "escalation": RuleBasedRouter([SHOPCO_ROUTES[0]], threshold=0.3),
        }

    def classify_l1(self, message: str) -> str:
        msg = message.lower()
        for domain, keywords in self.l1_rules.items():
            if domain == "general":
                continue
            if any(kw in msg for kw in keywords):
                return domain
        return "general"

    def handle(self, message: str) -> dict[str, Any]:
        l1 = self.classify_l1(message)
        if l1 in self.l2_routers:
            decision, result = self.l2_routers[l1].route_and_handle(message)
            return {"l1": l1, "l2": decision.route_id, "reply": result.reply}
        return {"l1": l1, "l2": None, "reply": handle_fallback(message, RouteDecision("fallback", 0, "", {})).reply}


hier = HierarchicalRouter()
for msg in ["Refund ORD-1001 please", "Get me a manager"]:
    out = hier.handle(msg)
    print(f"{msg!r} → L1={out['l1']} L2={out['l2']}")

---

## 8. ReleaseFlow Router

Engineering requests route to release check, hotfix, or general DevOps FAQ.

In [ ]:
def handle_release_check(message: str, decision: RouteDecision) -> HandlerResult:
    blockers = [f"{k}: {v['detail']}" for k, v in RELEASE_CHECKS.items() if v["status"] != "pass"]
    ver = re.search(r"v\d+\.\d+\.\d+", message)
    version = ver.group() if ver else "unknown"
    if blockers:
        return HandlerResult("release_check", f"NO-GO for {version}: {'; '.join(blockers)}")
    return HandlerResult("release_check", f"GO for {version}: all checks passed.")


def handle_hotfix(message: str, decision: RouteDecision) -> HandlerResult:
    return HandlerResult("hotfix", "Hotfix path: expedited review, skip non-critical checks, require on-call approval.")


def handle_devops_faq(message: str, decision: RouteDecision) -> HandlerResult:
    return HandlerResult("devops_faq", "See runbook: deploy.md for standard staging → prod promotion.")


RELEASE_ROUTES = [
    Route("release_check", "Go/no-go", handle_release_check, ["ship", "release", "deploy", "prod", "v"]),
    Route("hotfix", "Emergency patch", handle_hotfix, ["hotfix", "urgent", "p0", "incident"]),
    Route("devops_faq", "Runbooks", handle_devops_faq, ["how do i", "runbook", "pipeline"]),
]

release_router = RuleBasedRouter(RELEASE_ROUTES, threshold=0.35)
for msg in ["Can we ship v2.4.1 to prod?", "Need urgent hotfix for P0 outage", "How do I run the pipeline?"]:
    d, r = release_router.route_and_handle(msg)
    print(f"[{d.route_id}] {r.reply}")

---

## 9. Route + Chain Composition

Routing picks the branch; **prompt chaining** can run **inside** a handler. The router stays thin.

In [ ]:
def refund_chain_handler(message: str, decision: RouteDecision) -> HandlerResult:
    """Refund route uses internal extract → enrich → draft steps."""
    oid = re.search(r"ORD-\d+", message.upper())
    steps = ["extract", "policy_lookup", "draft"]
    if not oid:
        return HandlerResult("refund", "Need order ID for refund chain.", {"steps": ["extract"]})
    order = ORDERS.get(oid.group())
    draft = f"Draft refund response for {oid.group()}: {POLICIES['refunds']}"
    if order and order["total_usd"] > 1000:
        draft += " ESCALATE."
    return HandlerResult("refund", draft, {"steps": steps, "chain": True})


chain_router = RuleBasedRouter([
    Route("refund", "Refund chain", refund_chain_handler, ["refund"]),
    Route("order_status", "Orders", handle_order_status, ["status", "ord-"]),
])
_, res = chain_router.route_and_handle("Refund ORD-1001")
print(res.reply)
print("Internal chain steps:", res.metadata.get("steps"))

---

## 10. Misroute Detection

Log when users immediately rephrase — a signal the router picked wrong. Offline heuristic: negative sentiment + fallback route.

In [ ]:
@dataclass
class RouteQualitySignal:
    likely_misroute: bool
    signal: str


def detect_misroute(first_decision: RouteDecision, follow_up: str) -> RouteQualitySignal:
    fu = follow_up.lower()
    if first_decision.route_id == "fallback" and any(w in fu for w in ("no", "wrong", "not what", "refund", "order")):
        return RouteQualitySignal(True, "user corrected after fallback")
    if "that's not" in fu or "i said" in fu:
        return RouteQualitySignal(True, "explicit correction")
    return RouteQualitySignal(False, "ok")


d1, _ = router.route_and_handle("Hello")
signal = detect_misroute(d1, "No, I meant refund for ORD-1001")
print(f"First route: {d1.route_id}, misroute likely: {signal.likely_misroute} ({signal.signal})")

---

## 11. Route Analytics

In [ ]:
def route_analytics(trace: list[dict[str, Any]]) -> dict[str, Any]:
    counts: dict[str, int] = {}
    confidences: list[float] = []
    for entry in trace:
        rid = entry["route"]
        counts[rid] = counts.get(rid, 0) + 1
        confidences.append(entry["confidence"])
    avg_conf = sum(confidences) / len(confidences) if confidences else 0
    return {"total": len(trace), "by_route": counts, "avg_confidence": round(avg_conf, 3)}


print(pretty(route_analytics(ROUTE_TRACE)))

---

## 12. When to Use Routing

| Good fit | Poor fit |
|----------|----------|
| Distinct intents with different tools/prompts | One FAQ, one answer style |
| Support triage (billing vs shipping vs sales) | Continuous tool exploration |
| SLA differs by request type | Subtle nuance needing multi-step reasoning |
| Pre-filter before expensive agent | Single prompt chain covers all cases |

---

## 13. Anti-Patterns

| Anti-pattern | Problem | Fix |
|--------------|---------|-----|
| 20 routes with overlapping keywords | Misroutes | Merge routes or use hierarchical routing |
| No fallback | Wrong handler on ambiguous input | Confidence threshold + clarify |
| Router calls LLM + full agent each time | Double cost | Route once, then specialized cheap path |
| One handler does everything | Routing theater | Distinct tools per route |
| Ignore misroute signals | Drift in production | Log corrections, retrain classifier |
| Route after long chain | Late branching wastes steps | Route **first** |

---

## 14. Check Your Understanding

1. How is **routing** different from **prompt chaining**?
2. Why route **before** running specialized handlers?
3. What is the purpose of a **confidence threshold**?
4. When is **hierarchical routing** useful?
5. How can a route **internally** use a prompt chain?
6. What signal suggests a **misroute**?
7. Name two ShopCo routes and two ReleaseFlow routes.

---

## 15. Optional — Live LLM Classifier

Set `USE_LIVE_LLM = True` to classify with a model, then dispatch to the same handlers.

In [ ]:
if USE_LIVE_LLM:
    import os
    from openai import OpenAI

    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
    msg = "I need a refund for order ORD-1001"
    routes_desc = ", ".join(r.route_id for r in SHOPCO_ROUTES)
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": f"Classify into exactly one route: {routes_desc}, or fallback. Reply JSON: route_id, confidence."},
            {"role": "user", "content": msg},
        ],
        response_format={"type": "json_object"},
    )
    parsed = json.loads(resp.choices[0].message.content or "{}")
    rid = parsed.get("route_id", "fallback")
    handler = router._handlers.get(rid, handle_fallback)
    result = handler(msg, RouteDecision(rid, float(parsed.get("confidence", 0.8)), "llm"))
    print(result.reply)
else:
    print("Offline mode — rule-based routing with ShopCo and ReleaseFlow handlers.")

---

## 16. Summary

Routing **classifies once** and **dispatches** to a specialized handler:

- **Rule-based** routers use keywords and patterns; add **confidence thresholds** and **fallback**
- **Hierarchical** routing separates coarse domain from fine intent
- Each route owns focused logic; routes can wrap internal prompt chains
- **ShopCo** routes order status, policy, refund, and escalation; **ReleaseFlow** routes release checks and hotfixes

Route first, specialize second — keep the router thin and handlers testable.